# GPT-2 Text Generation & Fine-Tuning - Topic: Computer
# Sampling methods: Greedy to Top-P (nucleus)

In [41]:
# Step 1: Imports
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

In [42]:
# Step 2 – Config & Device Setup

BLOCK_SIZE = 32          # Max tokens per sentence
BATCH_SIZE = 2
EPOCHS = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "gpt2_computer_output"

In [43]:
# Step 3: Tiny Computer/dataset
texts = [
    "Computers are machines that process data.",
    "A mobile phone can run applications.",
    "Artificial Intelligence is the future.",
    "Python is a popular programming language.",
    "Hardware and software are parts of a computer."
]

In [44]:
# Step 4: Load GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Ensure padding token exists

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))  # Adjust embeddings if tokenizer changed
model.to(DEVICE)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [45]:
# Step 5: Tokenize each sentence
train_dataset = []
for text in texts:
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=BLOCK_SIZE, return_tensors="pt")
    train_dataset.append({
        "input_ids": enc["input_ids"][0],
        "attention_mask": enc["attention_mask"][0],
        "labels": enc["input_ids"][0]
    })


In [46]:
# Step 6: Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [47]:
# Step 7: Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    save_total_limit=1,
    logging_steps=1,
    report_to="none"
)


In [48]:
# Step 8: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)


In [49]:
# Step 9: Text Generation
prompt = "Computers can"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

# 1) Greedy Generation
greedy_output = model.generate(
    **inputs,
    max_length=50,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)
print("\n=== Greedy Output ===")
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

# 2) Top-p (Nucleus) Sampling
top_p_output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    num_return_sequences=2,
    pad_token_id=tokenizer.eos_token_id
)
print("\n=== Top-p Outputs ===")
for i, out in enumerate(top_p_output):
    print(f"--- Sample {i+1} ---")
    print(tokenizer.decode(out, skip_special_tokens=True))



=== Greedy Output ===
Computers can be used to create a new computer, but they can't be used to create a new computer.

The computer can't be used to create a new computer. The computer can't be used to create a new computer. The

=== Top-p Outputs ===
--- Sample 1 ---
Computers can use the 'sparse' command line option to parse a file.

The 'dont_parse' command line option can be used to bypass the need to parse a file. This option is only available for Linux.

--- Sample 2 ---
Computers can be a big win for Google, but it also means they're also trying to keep their business in line with the Internet.

Google's latest venture into artificial intelligence, called DeepMind, is an effort to build a new type


In [50]:
# Step 10a: Greedy Search
prompt = "Computers can"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

greedy_output = model.generate(
    **inputs,
    max_length=50,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

print("=== Greedy Search ===")
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))


=== Greedy Search ===
Computers can be used to create a new computer, but they can't be used to create a new computer.

The computer can't be used to create a new computer. The computer can't be used to create a new computer. The


In [51]:
# Step 10b: Beam Search
beam_output = model.generate(
    **inputs,
    max_length=50,
    num_beams=5,
    num_return_sequences=3,
    early_stopping=True,
    pad_token_id=tokenizer.eos_token_id
)

print("=== Beam Search ===")
for i, out in enumerate(beam_output):
    print(f"--- Beam {i+1} ---")
    print(tokenizer.decode(out, skip_special_tokens=True))


=== Beam Search ===
--- Beam 1 ---
Computers can't do anything about it.

I'm not sure if this is a good thing or a bad thing.

I'm not sure if this is a good thing or a bad thing.

I'm not sure if
--- Beam 2 ---
Computers can't do anything about it.

I'm not sure if this is a good thing or a bad thing.

I'm not sure if this is a good thing or a bad thing. I'm not sure if this is
--- Beam 3 ---
Computers can't do anything about it.

I'm not sure if this is a good thing or a bad thing.

I'm not sure if this is a good thing or a bad thing. I'm not sure if it's


In [52]:
# Step 10c: Random Sampling
random_output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_k=0,
    temperature=0.8,
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id
)

print("=== Random Sampling ===")
for i, out in enumerate(random_output):
    print(f"--- Sample {i+1} ---")
    print(tokenizer.decode(out, skip_special_tokens=True))


=== Random Sampling ===
--- Sample 1 ---
Computers can be used to make more powerful engines commonly used in modern post-processing applications. Two examples came from theoretical work: frequency-limiting algorithms and low-level hardware junction (LMI) techniques.

CCIP gives a
--- Sample 2 ---
Computers can process Web2D objects at scale, but all GPUs and CPUs need to be able to see through the pixels to render Web2D objects. For many applications though, rendering Web2D objects requires the client to have some sort of
--- Sample 3 ---
Computers can have a real-time clock running, it's amazing to see it run as fast as Speedrun has. The microcomputer uses a 12MHz core, with 6 blocks of myRAM, and uses the same LPDDR3 logic


In [53]:
# Step 10d: Top-K Sampling
top_k_output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_k=50,
    temperature=0.8,
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id
)

print("=== Top-K Sampling ===")
for i, out in enumerate(top_k_output):
    print(f"--- Sample {i+1} ---")
    print(tokenizer.decode(out, skip_special_tokens=True))


=== Top-K Sampling ===
--- Sample 1 ---
Computers can be a little over-powered, though. The only problem is that it works in a way that's difficult for a user to overcome in terms of sheer power.

The new device, which will be manufactured in the UK,
--- Sample 2 ---
Computers can also be used to read and write data. A machine with some CPU power can run Windows XP.

Intel I/O with Intel I/O

Another important feature of the processor is the I/O bus. This
--- Sample 3 ---
Computers can be moved from one storage location to another. You can use these to move your data from one storage location to another.

This is particularly useful for caching data for some datasets, and for storing things like the data for an upcoming


In [54]:
# Step 10e: Top-P Sampling
top_p_output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id
)

print("=== Top-P Sampling ===")
for i, out in enumerate(top_p_output):
    print(f"--- Sample {i+1} ---")
    print(tokenizer.decode(out, skip_special_tokens=True))


=== Top-P Sampling ===
--- Sample 1 ---
Computers can also perform tasks like retrieving data from the Internet.

The new devices are being developed to act as servers and devices for the Internet. But while the computers can communicate with the computers on the network, the Internet itself will not be
--- Sample 2 ---
Computers can handle many kinds of tasks, from a high-performance, high-speed, and high-energy computing platform.

This is why it's important to make sure that your system is properly configured.

A simple test to
--- Sample 3 ---
Computers can be used to perform calculations, and they can even be used for data mining. The most popular method is by using a computer to build software for processing, but you can also use a computer to do other things.

Here's


In [ ]:
# Step 10f: Top-K + Top-P Sampling
top_kp_output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.8,
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id
)

print("=== Top-K + Top-P Sampling ===")
for i, out in enumerate(top_kp_output):
    print(f"--- Sample {i+1} ---")
    print(tokenizer.decode(out, skip_special_tokens=True))
